In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import json
import ast
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ExponentialLR
from torch.nn.utils.rnn import pad_sequence
from torch.utils.tensorboard import SummaryWriter

In [2]:
nel_labels = pd.read_csv('data/data_train.csv')['nel'].to_numpy()[0]

In [3]:
nel_labels = ast.literal_eval(nel_labels)

In [4]:
nel_labels

[0, 4, 5, 17]

In [5]:
nel_label_list = np.zeros(18)
for i in nel_labels:
    nel_label_list[i] = 1

In [6]:
torch.tensor(nel_label_list, dtype=torch.float)

tensor([1., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])

In [7]:
class CustomDataset(Dataset):
    def __init__(self, mode):
        self.mode = mode
        if mode == 'train':
            with open('data/ScanRefer_filtered_train_with_id.json', 'r') as file:
                data = json.load(file)
            self.labels = pd.read_csv('data/data_train.csv')['class'].to_numpy()
            self.nel_labels = pd.read_csv('data/data_train.csv')['nel'].to_numpy()
        else:
            with open('data/ScanRefer_filtered_val_with_id.json', 'r') as file:
                data = json.load(file)
            self.labels = pd.read_csv('data/data_val.csv')['class'].to_numpy()
            self.nel_labels = pd.read_csv('data/data_val.csv')['nel'].to_numpy()
        self.num_samples = len(data)
        
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        if self.mode == 'train':
            loaded_tensor = torch.load('data/contextual_train/{}.pt'.format(idx))
        else:
            loaded_tensor = torch.load('data/contextual_val/{}.pt'.format(idx))
        label = self.labels[idx]
        nel_label = ast.literal_eval(self.nel_labels[idx])
        nel_label_list = np.zeros(18)
        for i in nel_label:
            nel_label_list[i] = 1
        # labels = torch.tensor(label, dtype=torch.long)
        nel_labels = torch.tensor(nel_label_list, dtype=torch.float)
        return loaded_tensor, label, nel_labels
        
def custom_collate_fn(batch):
    # Unpack batch into separate lists of inputs and labels
    inputs, labels, nel_labels = zip(*batch)
    # inputs: tuple of [L_i, 768] → pad to [B, max_len, 768]
    padded_inputs = pad_sequence(inputs, batch_first=True)  # Pads on dim=0 (sequence length)

    # Collect lengths (useful for masking or packing)
    lengths = torch.tensor([x.size(0) for x in inputs], dtype=torch.long)

    # Convert labels to tensor
    labels = torch.tensor(labels, dtype=torch.long)
    nel_labels = torch.stack(nel_labels)
    return padded_inputs, labels, nel_labels, lengths

In [26]:
class Classifier(nn.Module):
    def __init__(self, num_classes=18):
        super(Classifier, self).__init__()

        self.gru = nn.GRU(
            input_size=768,
            hidden_size=512,
            batch_first=True,
            bidirectional=True
        )
        for name, param in self.gru.named_parameters():
            param.requires_grad = True
        self.nel_classifier = nn.Sequential(
            # nn.Linear(1024, 512),
            # # nn.BatchNorm1d(512),
            # nn.ReLU(inplace=True),
            # nn.Dropout(0.1),
            # nn.Linear(512, 256),
            # # nn.BatchNorm1d(256),
            # nn.ReLU(inplace=True),
            # nn.Dropout(0.1),
            nn.Linear(1024, num_classes)
            )
        self.classifier = nn.Sequential(
            # nn.Linear(1024, 1024),
            # nn.BatchNorm1d(1024),
            # nn.ReLU(inplace=True),
            # nn.Dropout(0.1),
            # nn.Linear(1024, 256),
            # nn.BatchNorm1d(256),
            # nn.ReLU(inplace=True),
            # nn.Dropout(0.1),
            nn.Linear(1024, num_classes)
            )
    
    def forward(self, padded_tensor, lengths):
        batch_size = padded_tensor.shape[0]
        gru_out = []
        for i in range(batch_size):
            input_tensor = padded_tensor[i]
            input_tensor = input_tensor[:lengths[i], :].unsqueeze(0)
            _, hidden = self.gru(input_tensor)
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
            gru_out.append(hidden)
        tensor = torch.stack(gru_out).squeeze(1)
        # c_x = self.audio_classifier(x)
        # c_y = self.nel_classifier(x)
        # c_y = torch.sigmoid(c_y) ?
        x = self.classifier(tensor)
        y = self.nel_classifier(tensor)
        return x, y  #

In [27]:
# Instantiate the dataset
train_dataset = CustomDataset(mode='train')
val_dataset = CustomDataset(mode='val')
# Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=8, collate_fn= custom_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True, num_workers=8, collate_fn= custom_collate_fn)

In [37]:
len(val_loader)

298

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [45]:
model = Classifier().to(device)
# checkpoint = torch.load('model_state.pt')  # or 'model.pt', depending on your file
# model.load_state_dict(checkpoint)

In [46]:
loss_fn1 = nn.CrossEntropyLoss().to(device)
loss_fn2 = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = ExponentialLR(optimizer, gamma=0.99)
epochs = 10

In [47]:
iter = 0
writer = SummaryWriter(log_dir='runs/classifier_experiment')
for epoch in range(epochs):
    model.train()
    print('training at epoch: {}'.format(epoch))
    scheduler.step()
    for padded_tensor, labels, nel_labels, lengths in train_loader:
        iter += 1
        padded_tensor = padded_tensor.to(device)
        labels = labels.to(device)
        nel_labels = nel_labels.to(device)
        lengths = lengths.to(device)
        x, y = model(padded_tensor, lengths)
        # print(outputs.requires_grad)
        
        optimizer.zero_grad()
        loss_1 = loss_fn1(x, labels)
        loss_2 = loss_fn2(y, nel_labels)
        loss = loss_1 + 2*loss_2
        loss.backward()
        optimizer.step()
        if iter % 10 == 0:
            writer.add_scalar('Loss1/train', loss_1, iter)
            writer.add_scalar('Loss2/train', loss_2, iter)
            print('loss 1: ', loss_1)
            print('loss 2: ', loss_2)

    model.eval()
    print('eval ...')
    correct = 0
    total = 0
    total_precision = 0
    total_recall = 0
    total_f1 = 0
    
    for padded_tensor, labels, nel_labels, lengths in val_loader:
        padded_tensor = padded_tensor.to(device)
        labels = labels.to(device)
        nel_labels = nel_labels.to(device)
        lengths = lengths.to(device)
        x, y = model(padded_tensor, lengths)
        
        preds = x.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
        probs = torch.sigmoid(y)
        preds = (probs > 0.5).int()
        nel_labels = nel_labels.int()
        TP = (preds & nel_labels).sum(dim=0)
        FP = (preds & (1 - nel_labels)).sum(dim=0)
        FN = ((1 - preds) & nel_labels).sum(dim=0)
        
        # Add epsilon to avoid division by zero
        eps = 1e-8
        
        precision = TP / (TP + FP + eps)
        recall    = TP / (TP + FN + eps)
        f1        = 2 * precision * recall / (precision + recall + eps)
    
        total_precision += precision
        total_recall += recall
        total_f1 += f1
    accuracy = correct / total
    total_precision = total_precision / len(val_loader)
    total_recall = total_recall / len(val_loader)
    total_f1 = total_f1 / len(val_loader)
    print('accuracy: ', accuracy)
    print('total_precision: ', total_precision)
    print('total_recall: ', total_recall)
    print('total_f1: ', total_f1)
    
    print('mean_precision: ', torch.mean(total_precision))
    print('mean_recall: ', torch.mean(total_recall))
    print('mean_f1: ', torch.mean(total_f1))

training at epoch: 0


/home/duccd/miniconda3/envs/test/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:131: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


loss 1:  tensor(2.3671, device='cuda:0', grad_fn=<NllLossBackward0>)
loss 2:  tensor(0.2736, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
loss 1:  tensor(2.5283, device='cuda:0', grad_fn=<NllLossBackward0>)
loss 2:  tensor(0.2864, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
loss 1:  tensor(1.8719, device='cuda:0', grad_fn=<NllLossBackward0>)
loss 2:  tensor(0.2620, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
loss 1:  tensor(1.8824, device='cuda:0', grad_fn=<NllLossBackward0>)
loss 2:  tensor(0.2536, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
loss 1:  tensor(1.9921, device='cuda:0', grad_fn=<NllLossBackward0>)
loss 2:  tensor(0.2807, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
loss 1:  tensor(1.9652, device='cuda:0', grad_fn=<NllLossBackward0>)
loss 2:  tensor(0.2395, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
loss

In [44]:
model.eval()
print('eval ...')
correct = 0
total = 0
total_precision = 0
total_recall = 0
total_f1 = 0

for padded_tensor, labels, nel_labels, lengths in val_loader:
    padded_tensor = padded_tensor.to(device)
    labels = labels.to(device)
    nel_labels = nel_labels.to(device)
    lengths = lengths.to(device)
    x, y = model(padded_tensor, lengths)
    
    preds = x.argmax(dim=1)
    correct += (preds == labels).sum().item()
    total += labels.size(0)

    probs = torch.sigmoid(y)
    preds = (probs > 0.5).int()
    nel_labels = nel_labels.int()
    TP = (preds & nel_labels).sum(dim=0)
    FP = (preds & (1 - nel_labels)).sum(dim=0)
    FN = ((1 - preds) & nel_labels).sum(dim=0)
    
    # Add epsilon to avoid division by zero
    eps = 1e-8
    
    precision = TP / (TP + FP + eps)
    recall    = TP / (TP + FN + eps)
    f1        = 2 * precision * recall / (precision + recall + eps)

    total_precision += precision
    total_recall += recall
    total_f1 += f1
accuracy = correct / total
total_precision = total_precision / len(val_loader)
total_recall = total_recall / len(val_loader)
total_f1 = total_f1 / len(val_loader)
print('accuracy: ', accuracy)
print('total_precision: ', total_precision)
print('total_recall: ', total_recall)
print('total_f1: ', total_f1)

print('mean_precision: ', torch.mean(total_precision))
print('mean_recall: ', torch.mean(total_recall))
print('mean_f1: ', torch.mean(total_f1))

eval ...
accuracy:  0.9510938157341187
total_precision:  tensor([0.9629, 0.8780, 0.9972, 0.7359, 0.9987, 0.9482, 0.9370, 0.3638, 0.4362,
        0.6934, 0.9611, 0.5034, 0.0000, 0.1611, 0.6829, 0.8255, 0.2148, 0.9364],
       device='cuda:0')
total_recall:  tensor([0.9272, 0.8670, 0.9888, 0.7029, 0.9899, 0.9221, 0.8975, 0.3445, 0.4010,
        0.6864, 0.9547, 0.4670, 0.0000, 0.1566, 0.6534, 0.7900, 0.2120, 0.9252],
       device='cuda:0')
total_f1:  tensor([0.9409, 0.8652, 0.9926, 0.7097, 0.9939, 0.9319, 0.9127, 0.3487, 0.4126,
        0.6864, 0.9568, 0.4789, 0.0000, 0.1570, 0.6636, 0.8030, 0.2107, 0.9294],
       device='cuda:0')
mean_precision:  tensor(0.6798, device='cuda:0')
mean_recall:  tensor(0.6603, device='cuda:0')
mean_f1:  tensor(0.6663, device='cuda:0')


In [1]:
import torch

In [2]:
a = torch.rand(3, 4)

In [3]:
a

tensor([[0.0618, 0.5544, 0.1881, 0.3306],
        [0.6878, 0.1788, 0.4234, 0.8215],
        [0.8869, 0.3372, 0.5221, 0.9872]])

In [6]:
a.unsqueeze(0).shape

torch.Size([1, 3, 4])